# Linear Regression

Learn this model in order: **setup → baseline → initial model → tuning → interpretation**.

We keep all 395 students, including 38 recorded zero grades. The same 316 training students and five folds are used in every notebook. The 79 test students belong to final evaluation in Notebook 04.

There are two clearly separated scenarios: without G1/G2 (main) and with both grades (later stage). Input timing, particularly absences, still needs verification. MAE remains the selection metric. The test results have already been inspected: rerunning the unchanged experiment reproduces it, rather than creating a new unseen test.

Results and explanations stay in this notebook. Only fitted pipelines are saved to `models/` for the comparison and evaluation notebooks.

## 1 Setup

Shared helpers in `src/model_utils.py` handle the split and repeated metric calculations. Model settings, fitting and tuning remain visible here. Preprocessing is always fitted inside the training fold.

MAE/RMSE measure grade-point error; R² describes squared-error performance. Accuracy, precision, recall, F1 and specificity evaluate the derived **predicted G3 < 10** flag against **actual G3 < 10**. No rounding or threshold tuning is used.

In [1]:
from pathlib import Path
import sys
import joblib
import pandas as pd
from IPython.display import display
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import cross_validate, GridSearchCV

PROJECT_ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents]
    if (folder / "data/student-mat.csv").is_file())
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from model_utils import (
    load_data, model_pipeline, preprocessing_for,
    CV, SCORING, METRICS, fold_results, cv_summary, pooled_risk,
)

df, train_ids, test_ids = load_data(PROJECT_ROOT)
y_train = df.loc[train_ids, "G3"]
X_no_grades = df.loc[train_ids].drop(columns=["G1", "G2", "G3"])
X_with_grades = df.loc[train_ids].drop(columns="G3")
print("Training:", len(train_ids), "| Held-out test:", len(test_ids))
print("Inputs without / with G1,G2:", X_no_grades.shape[1], "/", X_with_grades.shape[1])

Training: 316 | Held-out test: 79
Inputs without / with G1,G2: 30 / 32


In [2]:
MODEL_NAME = 'Linear Regression'
MODEL_SLUG = 'linear_regression'
SCALE_NUMERIC = True

from sklearn.linear_model import LinearRegression
INITIAL_ESTIMATOR = LinearRegression()
MODEL_PARAM_GRID = {}

## 2 Constant baselines

Mean and median baselines learn their constant from each fold's training labels. They ignore the inputs, so one set of scores applies to both scenarios. Precision is undefined when nobody is flagged; missing values are left visible rather than replaced with perfect scores.

In [3]:
baseline_folds = {}
for strategy in ["mean", "median"]:
    scores = cross_validate(DummyRegressor(strategy=strategy), X_no_grades, y_train,
        cv=CV, scoring=SCORING, error_score="raise")
    baseline_folds[strategy] = fold_results(scores)

display(pd.DataFrame({name: folds[METRICS].mean()
    for name, folds in baseline_folds.items()}).T.round(4))

,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity
mean,3.4029,4.5635,-0.0350,0.6744,NaN,0.0,0.0,1.0
median,3.3403,4.5931,-0.0519,0.6744,NaN,0.0,0.0,1.0


## 3 Without G1 and G2

This is the main prediction scenario. Excluding grades does not by itself prove that all other inputs are available early.

### Initial model

Start with the configuration in the setup and original absences. The table shows fold means, standard deviations and how many folds have defined metrics.

In [4]:
initial_no_grades = model_pipeline(INITIAL_ESTIMATOR,
    scale_numeric=SCALE_NUMERIC, include_prior_grades=False)
initial_scores_no_grades = cross_validate(initial_no_grades, X_no_grades, y_train,
    cv=CV, scoring=SCORING, return_train_score=True, n_jobs=-1, error_score="raise")
initial_folds_no_grades = fold_results(initial_scores_no_grades)
display(cv_summary(initial_folds_no_grades).round(4))

,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity
mean,3.5227,4.6070,-0.0924,0.6422,0.4668,0.6023,0.5178,0.6673
std,0.4269,0.3657,0.2497,0.0430,0.1326,0.0785,0.0877,0.0722
defined folds,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000


### Tune using training cross-validation

Search the model settings from the setup and original versus log-transformed absences. Linear Regression searches only that preprocessing choice; it has no Ridge-style alpha. Select by MAE, then refit the chosen pipeline on all 316 training students. Classification metrics are supporting evidence and do not change selection.

In [5]:
grid_no_grades = dict(MODEL_PARAM_GRID)
grid_no_grades["preprocessing"] = [
    preprocessing_for(use_log=use_log, scale_numeric=SCALE_NUMERIC,
        include_prior_grades=False)
    for use_log in [False, True]
]
search_no_grades = GridSearchCV(initial_no_grades, grid_no_grades,
    scoring=SCORING, refit="MAE", cv=CV, return_train_score=True,
    n_jobs=-1, error_score="raise")
search_no_grades.fit(X_no_grades, y_train)
best_no_grades = search_no_grades.best_estimator_
tuned_folds_no_grades = fold_results(search_no_grades.cv_results_, search_no_grades.best_index_)

print("Model settings:", best_no_grades.named_steps["model"])
print("Log absences:", best_no_grades.named_steps["preprocessing"].named_steps["absence_transform"].kw_args["use_log"])
display(cv_summary(tuned_folds_no_grades).round(4))

,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity
mean,3.4763,4.4615,-0.0308,0.6613,0.4844,0.6218,0.5382,0.6834
std,0.3459,0.3408,0.2687,0.0227,0.1072,0.0843,0.0728,0.0549
defined folds,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000


Model settings: LinearRegression()
Log absences: True


### Compare and inspect errors

Compare regression and risk metrics on the same folds. The training/validation MAE gap helps identify overfitting. Pooled risk metrics come from adding the five fold confusion matrices; pooled rates can differ from fold averages. FN means missed at-risk students and FP means unnecessary flags.

The selected settings were chosen using these same folds, so their scores may be optimistic. They are not independent final-test or nested-CV estimates.

In [6]:
comparison_no_grades = pd.DataFrame({
    "Mean baseline": baseline_folds["mean"][METRICS].mean(),
    "Median baseline": baseline_folds["median"][METRICS].mean(),
    "Initial model": initial_folds_no_grades[METRICS].mean(),
    "Tuned model": tuned_folds_no_grades[METRICS].mean(),
}).T
display(comparison_no_grades.round(4))

training_no_grades = fold_results(search_no_grades.cv_results_, search_no_grades.best_index_, partition="train")
print("Tuned training MAE:", round(training_no_grades.MAE.mean(), 4))
print("Tuned validation MAE:", round(tuned_folds_no_grades.MAE.mean(), 4))
display(pd.DataFrame({"Initial model": pooled_risk(initial_folds_no_grades),
    "Tuned model": pooled_risk(tuned_folds_no_grades)}).T.round(4))
display(tuned_folds_no_grades.round(4))

,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity
Mean baseline,3.4029,4.5635,-0.0350,0.6744,NaN,0.0000,0.0000,1.0000
Median baseline,3.3403,4.5931,-0.0519,0.6744,NaN,0.0000,0.0000,1.0000
Initial model,3.5227,4.6070,-0.0924,0.6422,0.4668,0.6023,0.5178,0.6673
Tuned model,3.4763,4.4615,-0.0308,0.6613,0.4844,0.6218,0.5382,0.6834


,Accuracy,Precision,Recall,F1,Specificity,TP,FP,FN,TN
Initial model,0.6424,0.4627,0.6019,0.5232,0.6620,62.0,72.0,41.0,141.0
Tuned model,0.6614,0.4848,0.6214,0.5447,0.6808,64.0,68.0,39.0,145.0


,Fold,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity,TN,FP,FN,TP
0,1,3.3903,4.4905,-0.4044,0.6875,0.6538,0.6071,0.6296,0.7500,27.0,9.0,11.0,17.0
1,2,3.7955,4.6287,0.2885,0.6508,0.4839,0.7143,0.5769,0.6190,26.0,16.0,6.0,15.0
2,3,3.0260,3.8739,-0.0685,0.6349,0.3913,0.5000,0.4390,0.6889,31.0,14.0,9.0,9.0
3,4,3.8521,4.7410,-0.1289,0.6508,0.3929,0.6875,0.5000,0.6383,30.0,17.0,5.0,11.0
4,5,3.3177,4.5735,0.1593,0.6825,0.5000,0.6000,0.5455,0.7209,31.0,12.0,8.0,12.0


Tuned training MAE: 2.8546
Tuned validation MAE: 3.4763


## 4 With G1 and G2

This is a later-stage comparison, usable only once both prior grades are known. It answers a different prediction-timing question.

### Initial model

Start with the configuration in the setup and original absences. The table shows fold means, standard deviations and how many folds have defined metrics.

In [7]:
initial_with_grades = model_pipeline(INITIAL_ESTIMATOR,
    scale_numeric=SCALE_NUMERIC, include_prior_grades=True)
initial_scores_with_grades = cross_validate(initial_with_grades, X_with_grades, y_train,
    cv=CV, scoring=SCORING, return_train_score=True, n_jobs=-1, error_score="raise")
initial_folds_with_grades = fold_results(initial_scores_with_grades)
display(cv_summary(initial_folds_with_grades).round(4))

,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity
mean,1.3927,1.9869,0.8039,0.8388,0.6858,0.9524,0.7949,0.7803
std,0.2576,0.4037,0.0354,0.0835,0.1205,0.0734,0.1016,0.1081
defined folds,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000


### Tune using training cross-validation

Search the model settings from the setup and original versus log-transformed absences. Linear Regression searches only that preprocessing choice; it has no Ridge-style alpha. Select by MAE, then refit the chosen pipeline on all 316 training students. Classification metrics are supporting evidence and do not change selection.

In [8]:
grid_with_grades = dict(MODEL_PARAM_GRID)
grid_with_grades["preprocessing"] = [
    preprocessing_for(use_log=use_log, scale_numeric=SCALE_NUMERIC,
        include_prior_grades=True)
    for use_log in [False, True]
]
search_with_grades = GridSearchCV(initial_with_grades, grid_with_grades,
    scoring=SCORING, refit="MAE", cv=CV, return_train_score=True,
    n_jobs=-1, error_score="raise")
search_with_grades.fit(X_with_grades, y_train)
best_with_grades = search_with_grades.best_estimator_
tuned_folds_with_grades = fold_results(search_with_grades.cv_results_, search_with_grades.best_index_)

print("Model settings:", best_with_grades.named_steps["model"])
print("Log absences:", best_with_grades.named_steps["preprocessing"].named_steps["absence_transform"].kw_args["use_log"])
display(cv_summary(tuned_folds_with_grades).round(4))

,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity
mean,1.3724,1.8829,0.8239,0.8610,0.7209,0.9606,0.8211,0.8066
std,0.2215,0.3591,0.0254,0.0694,0.1035,0.0458,0.0754,0.1124
defined folds,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000


Model settings: LinearRegression()
Log absences: True


### Compare and inspect errors

Compare regression and risk metrics on the same folds. The training/validation MAE gap helps identify overfitting. Pooled risk metrics come from adding the five fold confusion matrices; pooled rates can differ from fold averages. FN means missed at-risk students and FP means unnecessary flags.

The selected settings were chosen using these same folds, so their scores may be optimistic. They are not independent final-test or nested-CV estimates.

In [9]:
comparison_with_grades = pd.DataFrame({
    "Mean baseline": baseline_folds["mean"][METRICS].mean(),
    "Median baseline": baseline_folds["median"][METRICS].mean(),
    "Initial model": initial_folds_with_grades[METRICS].mean(),
    "Tuned model": tuned_folds_with_grades[METRICS].mean(),
}).T
display(comparison_with_grades.round(4))

training_with_grades = fold_results(search_with_grades.cv_results_, search_with_grades.best_index_, partition="train")
print("Tuned training MAE:", round(training_with_grades.MAE.mean(), 4))
print("Tuned validation MAE:", round(tuned_folds_with_grades.MAE.mean(), 4))
display(pd.DataFrame({"Initial model": pooled_risk(initial_folds_with_grades),
    "Tuned model": pooled_risk(tuned_folds_with_grades)}).T.round(4))
display(tuned_folds_with_grades.round(4))

,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity
Mean baseline,3.4029,4.5635,-0.0350,0.6744,NaN,0.0000,0.0000,1.0000
Median baseline,3.3403,4.5931,-0.0519,0.6744,NaN,0.0000,0.0000,1.0000
Initial model,1.3927,1.9869,0.8039,0.8388,0.6858,0.9524,0.7949,0.7803
Tuned model,1.3724,1.8829,0.8239,0.8610,0.7209,0.9606,0.8211,0.8066


,Accuracy,Precision,Recall,F1,Specificity,TP,FP,FN,TN
Initial model,0.8386,0.6806,0.9515,0.7935,0.7840,98.0,46.0,5.0,167.0
Tuned model,0.8608,0.7122,0.9612,0.8182,0.8122,99.0,40.0,4.0,173.0


,Fold,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity,TN,FP,FN,TP
0,1,1.2191,1.5748,0.8273,0.7812,0.6750,0.9643,0.7941,0.6389,23.0,13.0,1.0,27.0
1,2,1.6294,2.3401,0.8181,0.9048,0.7778,1.0000,0.8750,0.8571,36.0,6.0,0.0,21.0
2,3,1.2443,1.6627,0.8032,0.7937,0.5926,0.8889,0.7111,0.7556,34.0,11.0,2.0,16.0
3,4,1.1725,1.6350,0.8657,0.8889,0.6957,1.0000,0.8205,0.8511,40.0,7.0,0.0,16.0
4,5,1.5965,2.2020,0.8051,0.9365,0.8636,0.9500,0.9048,0.9302,40.0,3.0,1.0,19.0


Tuned training MAE: 1.125
Tuned validation MAE: 1.3724


## 5 Compare scenarios and save the fitted models

The score difference reflects the information available at different prediction times. Do not present the G1/G2 scenario as an early prediction. Save only the two complete fitted pipelines; tables and findings remain inside the notebook.

In [10]:
display(pd.DataFrame({
    "Without G1/G2": tuned_folds_no_grades[METRICS].mean(),
    "With G1/G2": tuned_folds_with_grades[METRICS].mean(),
}).T.round(4))

model_dir = PROJECT_ROOT / "models/experiments"
model_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(best_no_grades, model_dir / "linear_regression__no_prior_grades.joblib")
joblib.dump(best_with_grades, model_dir / "linear_regression__with_prior_grades.joblib")
print("Saved fitted pipelines. No experiment tables or reports are exported.")

,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity
Without G1/G2,3.4763,4.4615,-0.0308,0.6613,0.4844,0.6218,0.5382,0.6834
With G1/G2,1.3724,1.8829,0.8239,0.8610,0.7209,0.9606,0.8211,0.8066


Saved fitted pipelines. No experiment tables or reports are exported.
